# 🎯 Automated Candidate Interview & Evaluation System

This notebook demonstrates an AI-powered interview system using **AutoGen AgentChat**. Three specialised agents collaborate in a round-robin group chat to conduct a realistic mock interview, collect human answers, and deliver actionable feedback.

| Agent | Role |
|-------|------|
| **Interviewer** | Asks structured interview questions tailored to the role |
| **Candidate** | Human-in-the-loop proxy that accepts real user input |
| **Evaluator** | Career coach that critiques answers and suggests improvements |

## Learning Objectives
In this notebook, you will learn:
1. **Multi-Agent Orchestration** — How to wire up a `RoundRobinGroupChat` with AutoGen AgentChat
2. **Human-in-the-Loop** — Using `UserProxyAgent` to inject human input into an agent loop (scripted answers by default, your own answers on demand)
3. **Streaming Results** — Consuming `run_stream` to display interview progress in real time
4. **Termination Conditions** — Controlling conversation length with `TextMentionTermination`

## Prerequisites
- Dependencies installed with `uv sync` in this project's folder
- `GROQ_API_KEY` in this project's `.env` — the agents run on Groq-hosted Llama 3.3 70B through AutoGen's OpenAI-compatible model client, the same client `app.py` uses (no other keys needed)

---
## 📦 Step 1: Environment Setup

Load `GROQ_API_KEY` from this project's `.env` and import the AutoGen AgentChat primitives used throughout the notebook.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and API keys
# ============================================================================
import os
import warnings

from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.base import TaskResult
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads GROQ_API_KEY from this project's .env
load_dotenv(find_dotenv(usecwd=True))
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("✅ Environment ready")

### 🤖 Model Client Initialization

AutoGen agents take an AutoGen *model client*, not a LangChain chat model, so this notebook keeps AutoGen's `OpenAIChatCompletionClient` pointed at Groq's OpenAI-compatible endpoint — the same client the project's web app (`app.py`) uses. `model_info` tells AutoGen what the non-OpenAI model supports.

In [ ]:
# ============================================================================
# MODEL CLIENT INITIALIZATION: Groq-hosted Llama 3.3 70B via AutoGen
# ============================================================================
MODEL_NAME = "llama-3.3-70b-versatile"

model_client = OpenAIChatCompletionClient(
    model=MODEL_NAME,
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
    },
)

print(f"✅ Model client ready — using model: {MODEL_NAME}")

---
## 🤖 Part 1: Define the Interview Agents & Team

We create three agents — **Interviewer**, **Candidate**, and **Evaluator** — and wire them into a `RoundRobinGroupChat`. The conversation ends automatically after the interviewer says *TERMINATE*.

### Key Concepts:
- **AssistantAgent** — LLM-backed agent driven by a system prompt
- **UserProxyAgent** — Bridges human input into the agent loop through its `input_func`
- **RoundRobinGroupChat** — Cycles through participants in a fixed order
- **TextMentionTermination** — Stops the chat when a trigger word appears

In [ ]:
# ============================================================================
# AGENT DEFINITIONS: Interviewer, Candidate & Evaluator
# ============================================================================

async def build_interview_team(job_position: str = "AI Engineer", input_func=input):
    """Create and return a RoundRobinGroupChat team for the given role.

    input_func supplies the Candidate's answers: `input` for a person at the keyboard,
    or any function that takes the prompt string and returns an answer.
    """

    # --- Interviewer: asks 3 targeted questions then terminates ---
    interviewer = AssistantAgent(
        name="Interviewer",
        model_client=model_client,
        description=f"An AI agent that conducts interviews for a {job_position} position.",
        system_message=(
            f"You are a professional interviewer for a {job_position} position.\n"
            "Ask one clear question at a time and wait for the user to respond.\n"
            "Ignore the career coach's response — your job is only to ask questions.\n"
            "Tailor each follow-up based on the candidate's previous answer.\n"
            "Ask 3 questions total covering: technical skills, problem-solving, and cultural fit.\n"
            "After the 3rd question, say 'TERMINATE'.\n"
            "Keep every question under 50 words."
        ),
    )

    # --- Candidate: human-in-the-loop proxy ---
    candidate = UserProxyAgent(
        name="Candidate",
        description=f"A human proxy representing the candidate for a {job_position} position.",
        input_func=input_func,
    )

    # --- Evaluator: career coach giving real-time feedback ---
    evaluator = AssistantAgent(
        name="Evaluator",
        model_client=model_client,
        description=f"An AI career coach providing feedback for a {job_position} interview.",
        system_message=(
            f"You are a career coach specializing in {job_position} interviews.\n"
            "Provide constructive feedback on the candidate's responses and suggest improvements.\n"
            "After the interview, summarize performance and give actionable advice.\n"
            "Keep feedback under 100 words."
        ),
    )

    # --- Assemble the team ---
    termination_condition = TextMentionTermination(text="TERMINATE")

    team = RoundRobinGroupChat(
        participants=[interviewer, candidate, evaluator],
        termination_condition=termination_condition,
        max_turns=20,
    )

    return team

print("✅ build_interview_team() ready")

---
## 🔄 Part 2: Stream the Interview

Define an async generator that consumes the team's `run_stream` output and yields formatted messages as they arrive. This keeps the interview interactive rather than blocking until completion.

In [ ]:
# ============================================================================
# INTERVIEW STREAMING: Async Generator for Real-Time Output
# ============================================================================

async def run_interview(team: RoundRobinGroupChat):
    """Stream interview messages, yielding formatted strings as they arrive."""
    async for message in team.run_stream(
        task="Start the interview with the first question."
    ):
        if isinstance(message, TaskResult):
            yield f"✅ Interview completed — stop reason: {message.stop_reason}"
        else:
            yield f"🗣️ {message.source}: {message.content}"

print("✅ run_interview() ready")

---
## 🚀 Part 3: Run the Interview

Build the team for the desired role and kick off the interview. By default the **Candidate's** answers come from a short scripted list (`SCRIPTED_ANSWERS`), so the notebook runs top to bottom without a person at the keyboard. Set `INTERACTIVE = True` to type your own answers at the `input()` prompt each time it is the Candidate's turn.

> **Note**: Change `job_position` below to tailor questions to a different role (e.g., *"Data Scientist"*, *"Backend Engineer"*).

In [ ]:
# ============================================================================
# CANDIDATE INPUT: Scripted answers by default, your own answers on demand
# ============================================================================
INTERACTIVE = False  # set to True to answer the interviewer yourself via input()

SCRIPTED_ANSWERS = [
    "I build LLM applications in Python: retrieval pipelines with LangChain, evaluation "
    "harnesses, and FastAPI services that wrap them.",
    "When our RAG assistant started returning stale answers, I traced it to a missing "
    "re-index step, added a nightly job, and wrote a regression test with known questions.",
    "I like small teams that review each other's work openly. I share what I learn in short "
    "write-ups and ask for feedback early.",
]


def make_scripted_input(answers):
    """Return an input_func that replies with the next scripted answer on each call."""
    remaining = iter(answers)

    def scripted_input(prompt: str) -> str:
        # Fixed closing line once the scripted answers run out
        return next(remaining, "Thank you, I have nothing further to add.")

    return scripted_input


print(f"✅ Candidate input: {'interactive input()' if INTERACTIVE else f'{len(SCRIPTED_ANSWERS)} scripted answers'}")

In [ ]:
# ============================================================================
# RUN INTERVIEW: Build Team & Stream Conversation
# ============================================================================

job_position = "AI Engineer"

# A fresh scripted_input per run, so re-running this cell starts from the first answer
candidate_input = input if INTERACTIVE else make_scripted_input(SCRIPTED_ANSWERS)
team = await build_interview_team(job_position, input_func=candidate_input)

print(f"🤖 Starting interview for: {job_position}\n")

async for message in run_interview(team):
    print("-" * 70)
    print(message)

---
## 📝 Summary

In this notebook, we built an end-to-end **AI-powered mock interview system** using AutoGen AgentChat.

### 1. Environment Setup
- Loaded a **Groq-hosted Llama 3.3 70B** model via AutoGen's `OpenAIChatCompletionClient`

### 2. Agent Design
- **Interviewer** — asks 3 role-specific questions, then terminates
- **Candidate** — `UserProxyAgent` bridging human input (scripted answers by default, `input()` when `INTERACTIVE = True`)
- **Evaluator** — career coach delivering constructive feedback after each answer

### 3. Orchestration & Streaming
- `RoundRobinGroupChat` manages turn order across agents
- `TextMentionTermination` ends the loop when *TERMINATE* appears
- `run_stream` provides real-time output instead of blocking

### Next Steps
- **The project's app (`app.py`)**: runs the same three-agent team behind a FastAPI WebSocket chat UI (`uv run uvicorn app:app --reload`, see the README)
- Add more agents — e.g., a **Technical Assessor** that scores code-related answers
- Persist transcripts to a database for later review and analytics